# 01 · Time boundaries and pre-tournament snapshots

**NCAA probability forecasting** · Validation before feature selection

Notebook 00 establishes the data. This notebook makes the current feature pipeline's temporal contract visible. Notebook 02 measures feature contribution under this contract. No hidden variables from another notebook are required.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from march_mania.notebook_support import review_source, table, style

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "Open from the project directory"
CONFIG = json.loads((ROOT / "configs/feature_store.json").read_text())
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "figure.facecolor": "white", "axes.titleweight": "bold"})
style()

REVIEW, SUMMARY, SOURCE = review_source(ROOT, "data_review")
assert REVIEW is not None, "Run notebook 00 or march-audit first"
SPLITS = pd.read_csv(REVIEW / "splits.csv")
assert (SPLITS.training_max_season < SPLITS.validation_season).all()
assert (SPLITS.snapshot_cutoff < SPLITS.first_validation_day).all()
assert SPLITS.no_season_overlap.all()
display(Markdown(f"**Split evidence:** {SOURCE}. **Cutoff:** DayNum 132."))
table(SPLITS)

## Expanding-season validation

Each fold trains on entire earlier NCAA seasons and tests on the next selected development season. Mirrored training rows stay inside their original fold. Imputation and scaling are fitted only on training data. Current-season regular-season aggregates are allowed; current or future NCAA outcomes are not.

The five development seasons are **2016–2019 and 2021**. The previously reported 2022–2025 benchmark has already been consumed in earlier work; it must not be relabeled as untouched evidence for these new candidates.

In [ ]:
years = list(range(CONFIG["first_season"], max(CONFIG["validation_seasons"]) + 1))
folds = CONFIG["validation_seasons"]
matrix = np.zeros((len(folds), len(years)))
for row, valid in enumerate(folds):
    for column, year in enumerate(years):
        matrix[row, column] = 0 if year == 2020 or year > valid else (2 if year == valid else 1)
from matplotlib.colors import ListedColormap
fig, ax = plt.subplots(figsize=(11, 4), constrained_layout=True)
ax.imshow(matrix, cmap=ListedColormap(["#edf1f5", "#246b94", "#d98548"]), vmin=0, vmax=2, aspect="auto")
ax.set(xticks=range(len(years)), xticklabels=years, yticks=range(len(folds)),
       yticklabels=[f"Validate {year}" for year in folds], title="Blue: training · Orange: validation · Gray: unused")
for row in range(len(folds)):
    for col in range(len(years)):
        label = {0: "—", 1: "Train", 2: "Validate"}[matrix[row, col]]
        ax.text(col, row, label, ha="center", va="center", color="white" if matrix[row,col] else "#687484", fontsize=9)
plt.show()

## What a season-Y snapshot may know

| Input | Allowed information |
|---|---|
| Scores, box scores, venue, rest | Season-Y regular-season games through DayNum 132 |
| Tournament seeds | Published season-Y seeds available at the pre-tournament forecast |
| Massey | Publications through DayNum 132, with a 14-day staleness bound |
| Dynamic Elo | Earlier regular seasons plus legal season-Y regular games |
| Program and target history | NCAA outcomes from seasons strictly before Y |
| Preprocessing and fitted model | Earlier training seasons only |
| Validation labels | Held aside until probabilities are produced |

The inference date is after seed publication, before NCAA play-in games. Snapshots for later years support feature construction, not additional development scores.

In [ ]:
FEATURES, FEATURE_SUMMARY, FEATURE_SOURCE = review_source(ROOT, "feature_store")
if FEATURES is not None:
    coverage = pd.read_csv(FEATURES / "coverage.csv")
    display(Markdown(f"**Snapshot evidence:** {FEATURE_SOURCE}. "
                     f"**Team snapshots:** {FEATURE_SUMMARY['team_snapshots']:,}."))
    table(coverage)
else:
    display(Markdown("Run march-features after the data audit to build snapshots."))

## Why the notebooks can be run independently

The Python commands persist data manifests, season snapshots, features and fold results. Notebook 00 reviews the input evidence; 01 reviews the split evidence; 02 reviews the feature experiment. Running a later command does not require rerunning earlier notebook cells when its verified input artifacts already exist.

Continue to `02_feature_store_and_diagnostics.ipynb`. A new feature experiment compares the same physical games and preserves the completed earlier run. A feature earns inclusion through measured Brier improvement and stability, not through its name or complexity.